# Library DBMS — Step 4: SQL Schema

In [1]:
import sqlite3

DB_PATH = 'library.db'
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

cur.execute('PRAGMA foreign_keys = ON;')
print('Connected to', DB_PATH)

Connected to library.db


## 1 — Lookup / Reference Tables
Simple two-column lookup tables that are cleared by the two-attribute shortcut in Step 3.3.

In [2]:
cur.executescript('''
CREATE TABLE IF NOT EXISTS Type (
    type_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    type_name TEXT    NOT NULL UNIQUE          -- e.g. Book, DVD, Magazine
);

CREATE TABLE IF NOT EXISTS Acquisition_Status (
    acquisition_status_id INTEGER PRIMARY KEY AUTOINCREMENT,
    status_name           TEXT    NOT NULL UNIQUE  -- e.g. Purchased, Donated, Transferred
);

CREATE TABLE IF NOT EXISTS Copy_Status (
    copy_status_id INTEGER PRIMARY KEY AUTOINCREMENT,
    status_name    TEXT    NOT NULL UNIQUE          -- e.g. Available, Checked Out, Lost, Damaged
);

CREATE TABLE IF NOT EXISTS Member_Status (
    member_status_id INTEGER PRIMARY KEY AUTOINCREMENT,
    status_name      TEXT    NOT NULL UNIQUE        -- e.g. Active, Suspended, Expired
);

CREATE TABLE IF NOT EXISTS Request_Status (
    request_status_id INTEGER PRIMARY KEY AUTOINCREMENT,
    status_name       TEXT    NOT NULL UNIQUE       -- e.g. Open, In Progress, Resolved
);

CREATE TABLE IF NOT EXISTS Audience (
    audience_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    audience_type TEXT    NOT NULL UNIQUE           -- e.g. Children, Teens, Adults, Seniors
);
''')
print('Lookup tables created.')

Lookup tables created.


## 2 — Core Entity Tables

In [ ]:
cur.executescript('''
CREATE TABLE IF NOT EXISTS Library (
    library_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name       TEXT    NOT NULL,
    address    TEXT    NOT NULL
);

CREATE TABLE IF NOT EXISTS Personnel (
    personnel_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name   TEXT    NOT NULL,
    last_name    TEXT    NOT NULL,
    email        TEXT    NOT NULL UNIQUE,
    phone        TEXT,
    role         TEXT    NOT NULL,
    start_date   TEXT    NOT NULL,              -- stored as ISO-8601: YYYY-MM-DD
    salary       REAL    NOT NULL CHECK (salary >= 0),
    library_id   INTEGER NOT NULL REFERENCES Library(library_id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS Item (
    item_id               INTEGER PRIMARY KEY AUTOINCREMENT,
    title                 TEXT    NOT NULL,
    publication_year      INTEGER CHECK (publication_year BETWEEN 1000 AND 9999),
    genre                 TEXT,
    library_id            INTEGER NOT NULL REFERENCES Library(library_id) ON DELETE RESTRICT,
    type_id               INTEGER NOT NULL REFERENCES Type(type_id) ON DELETE RESTRICT,
    acquisition_status_id INTEGER NOT NULL REFERENCES Acquisition_Status(acquisition_status_id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS Author (
    author_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT    NOT NULL,
    last_name  TEXT    NOT NULL
);

CREATE TABLE IF NOT EXISTS Item_Copy (
    item_copy_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    date_obtained  TEXT    NOT NULL,            -- ISO-8601
    item_id        INTEGER NOT NULL REFERENCES Item(item_id) ON DELETE CASCADE,
    copy_status_id INTEGER NOT NULL REFERENCES Copy_Status(copy_status_id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS Member (
    member_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name       TEXT    NOT NULL,
    last_name        TEXT    NOT NULL,
    email            TEXT    NOT NULL UNIQUE,
    library_id       INTEGER NOT NULL REFERENCES Library(library_id) ON DELETE RESTRICT,
    member_status_id INTEGER NOT NULL REFERENCES Member_Status(member_status_id) ON DELETE RESTRICT
);

-- ISA specialisation — Volunteer IS-A Member
CREATE TABLE IF NOT EXISTS Volunteer (
    member_id    INTEGER PRIMARY KEY REFERENCES Member(member_id) ON DELETE CASCADE,
    start_date   TEXT    NOT NULL,
    availability TEXT
);

CREATE TABLE IF NOT EXISTS Room (
    room_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    room_name  TEXT    NOT NULL,
    capacity   INTEGER NOT NULL CHECK (capacity > 0),
    library_id INTEGER NOT NULL REFERENCES Library(library_id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS Event (
    event_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    title      TEXT    NOT NULL,
    date       TEXT    NOT NULL,               -- ISO-8601
    event_type TEXT    NOT NULL,
    room_id    INTEGER NOT NULL REFERENCES Room(room_id) ON DELETE RESTRICT
);
''')
print('Core entity tables created.')


## 3 — Relationship / Transaction Tables

In [ ]:
cur.executescript('''
-- Many-to-many: Item <-> Author
CREATE TABLE IF NOT EXISTS Written_By (
    item_id   INTEGER NOT NULL REFERENCES Item(item_id) ON DELETE CASCADE,
    author_id INTEGER NOT NULL REFERENCES Author(author_id) ON DELETE CASCADE,
    PRIMARY KEY (item_id, author_id)
);

-- Many-to-many: Event <-> Audience
CREATE TABLE IF NOT EXISTS Targets (
    event_id    INTEGER NOT NULL REFERENCES Event(event_id) ON DELETE CASCADE,
    audience_id INTEGER NOT NULL REFERENCES Audience(audience_id) ON DELETE RESTRICT,
    PRIMARY KEY (event_id, audience_id)
);

-- Many-to-many: Member <-> Event  (with extra attribute)
CREATE TABLE IF NOT EXISTS Registers_For (
    member_id         INTEGER NOT NULL REFERENCES Member(member_id) ON DELETE CASCADE,
    event_id          INTEGER NOT NULL REFERENCES Event(event_id) ON DELETE CASCADE,
    registration_date TEXT    NOT NULL,
    PRIMARY KEY (member_id, event_id)
);

-- Loan (transaction entity) -- history, so member/copy deletes are blocked, not cascaded
CREATE TABLE IF NOT EXISTS Loan (
    loan_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    loan_date     TEXT    NOT NULL,
    due_date      TEXT    NOT NULL,
    returned_date TEXT,                        -- NULL until returned
    member_id     INTEGER NOT NULL REFERENCES Member(member_id) ON DELETE RESTRICT,
    item_copy_id  INTEGER NOT NULL REFERENCES Item_Copy(item_copy_id) ON DELETE RESTRICT,
    CHECK (due_date >= loan_date),
    CHECK (returned_date IS NULL OR returned_date >= loan_date)
);

-- Fine — PK is (loan_id, fine_type) because one loan can incur multiple fine types
CREATE TABLE IF NOT EXISTS Fine (
    loan_id   INTEGER NOT NULL REFERENCES Loan(loan_id) ON DELETE CASCADE,
    fine_type TEXT    NOT NULL CHECK (fine_type IN ('Overdue', 'Damaged', 'Lost')),
    amount    REAL    NOT NULL CHECK (amount > 0),
    PRIMARY KEY (loan_id, fine_type)
);

-- Payment references Fine composite PK -- history, so blocked not cascaded
CREATE TABLE IF NOT EXISTS Payment (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date       TEXT    NOT NULL,
    amount     REAL    NOT NULL CHECK (amount > 0),
    loan_id    INTEGER NOT NULL,
    fine_type  TEXT    NOT NULL,
    FOREIGN KEY (loan_id, fine_type) REFERENCES Fine(loan_id, fine_type) ON DELETE RESTRICT
);

-- Donates — weak relationship; one copy donated by one member
CREATE TABLE IF NOT EXISTS Donates (
    item_copy_id  INTEGER PRIMARY KEY REFERENCES Item_Copy(item_copy_id) ON DELETE CASCADE,
    member_id     INTEGER NOT NULL REFERENCES Member(member_id) ON DELETE RESTRICT,
    donation_date TEXT    NOT NULL
);

-- Help_Request
CREATE TABLE IF NOT EXISTS Help_Request (
    help_request_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    request_date      TEXT    NOT NULL,
    description       TEXT    NOT NULL,
    member_id         INTEGER NOT NULL REFERENCES Member(member_id) ON DELETE RESTRICT,
    personnel_id      INTEGER          REFERENCES Personnel(personnel_id) ON DELETE SET NULL, -- nullable: unassigned
    request_status_id INTEGER NOT NULL REFERENCES Request_Status(request_status_id) ON DELETE RESTRICT
);
''')
print('Relationship / transaction tables created.')


## 4 — Indexes
Support common access patterns (FK lookups, date-range queries on loans/events).

In [5]:
cur.executescript('''
CREATE INDEX IF NOT EXISTS idx_personnel_library  ON Personnel(library_id);
CREATE INDEX IF NOT EXISTS idx_item_library       ON Item(library_id);
CREATE INDEX IF NOT EXISTS idx_item_copy_item     ON Item_Copy(item_id);
CREATE INDEX IF NOT EXISTS idx_member_library     ON Member(library_id);
CREATE INDEX IF NOT EXISTS idx_loan_member        ON Loan(member_id);
CREATE INDEX IF NOT EXISTS idx_loan_copy          ON Loan(item_copy_id);
CREATE INDEX IF NOT EXISTS idx_loan_due           ON Loan(due_date);
CREATE INDEX IF NOT EXISTS idx_event_room         ON Event(room_id);
CREATE INDEX IF NOT EXISTS idx_event_date         ON Event(date);
CREATE INDEX IF NOT EXISTS idx_payment_fine       ON Payment(loan_id, fine_type);
CREATE INDEX IF NOT EXISTS idx_helpreq_member     ON Help_Request(member_id);
CREATE INDEX IF NOT EXISTS idx_helpreq_personnel  ON Help_Request(personnel_id);
CREATE INDEX IF NOT EXISTS idx_helpreq_status     ON Help_Request(request_status_id);
''')
print('Indexes created.')

Indexes created.

## 5 — Triggers
Enforce business rules that CHECK constraints and FK constraints cannot express alone.

In [6]:
cur.executescript('''
-- ── T1: Auto-update Item_Copy status to "Checked Out" when a Loan is inserted ──────────
CREATE TRIGGER IF NOT EXISTS trg_loan_insert_copy_status
AFTER INSERT ON Loan
BEGIN
    UPDATE Item_Copy
    SET    copy_status_id = (
               SELECT copy_status_id FROM Copy_Status WHERE status_name = 'Checked Out'
           )
    WHERE  item_copy_id = NEW.item_copy_id;
END;


-- ── T2: Auto-update Item_Copy status back to "Available" when returned_date is set ─────
CREATE TRIGGER IF NOT EXISTS trg_loan_return_copy_status
AFTER UPDATE OF returned_date ON Loan
WHEN NEW.returned_date IS NOT NULL AND OLD.returned_date IS NULL
BEGIN
    UPDATE Item_Copy
    SET    copy_status_id = (
               SELECT copy_status_id FROM Copy_Status WHERE status_name = 'Available'
           )
    WHERE  item_copy_id = NEW.item_copy_id;
END;


-- ── T3: Prevent loaning a copy that is not currently "Available" ─────────────────────
CREATE TRIGGER IF NOT EXISTS trg_loan_prevent_unavailable
BEFORE INSERT ON Loan
BEGIN
    SELECT RAISE(ABORT, 'Item copy is not available for loan.')
    WHERE (
        SELECT cs.status_name
        FROM   Item_Copy ic
        JOIN   Copy_Status cs ON cs.copy_status_id = ic.copy_status_id
        WHERE  ic.item_copy_id = NEW.item_copy_id
    ) != 'Available';
END;


-- ── T4: Prevent suspended members from taking out loans ──────────────────────────────
CREATE TRIGGER IF NOT EXISTS trg_loan_prevent_suspended_member
BEFORE INSERT ON Loan
BEGIN
    SELECT RAISE(ABORT, 'Member account is not active.')
    WHERE (
        SELECT ms.status_name
        FROM   Member m
        JOIN   Member_Status ms ON ms.member_status_id = m.member_status_id
        WHERE  m.member_id = NEW.member_id
    ) != 'Active';
END;


-- ── T5: Payment total must not exceed Fine amount ────────────────────────────────────
CREATE TRIGGER IF NOT EXISTS trg_payment_no_overpay
BEFORE INSERT ON Payment
BEGIN
    SELECT RAISE(ABORT, 'Total payments would exceed the fine amount.')
    WHERE (
        SELECT COALESCE(SUM(amount), 0)
        FROM   Payment
        WHERE  loan_id   = NEW.loan_id
          AND  fine_type = NEW.fine_type
    ) + NEW.amount > (
        SELECT amount FROM Fine
        WHERE  loan_id   = NEW.loan_id
          AND  fine_type = NEW.fine_type
    );
END;


-- ── T6: Event must not be double-booked in the same room on the same date ────────────
CREATE TRIGGER IF NOT EXISTS trg_event_no_room_double_book
BEFORE INSERT ON Event
BEGIN
    SELECT RAISE(ABORT, 'Room is already booked for another event on this date.')
    WHERE EXISTS (
        SELECT 1 FROM Event
        WHERE  room_id = NEW.room_id
          AND  date    = NEW.date
    );
END;


-- ── T7: Registrations capped at room capacity ────────────────────────────────────────
CREATE TRIGGER IF NOT EXISTS trg_event_capacity_check
BEFORE INSERT ON Registers_For
BEGIN
    SELECT RAISE(ABORT, 'Event has reached its room capacity.')
    WHERE (
        SELECT COUNT(*) FROM Registers_For WHERE event_id = NEW.event_id
    ) >= (
        SELECT r.capacity
        FROM   Event e
        JOIN   Room  r ON r.room_id = e.room_id
        WHERE  e.event_id = NEW.event_id
    );
END;


-- ── T8: Volunteer must already be a Member ───────────────────────────────────────────
--  (enforced structurally via the FK on Volunteer.member_id → Member.member_id,
--   but this trigger adds a readable error message)
CREATE TRIGGER IF NOT EXISTS trg_volunteer_must_be_member
BEFORE INSERT ON Volunteer
BEGIN
    SELECT RAISE(ABORT, 'A volunteer must first be a registered member.')
    WHERE NOT EXISTS (
        SELECT 1 FROM Member WHERE member_id = NEW.member_id
    );
END;
''')
print('Triggers created.')

Triggers created.


## 6 — Seed Reference Data
Populate lookup tables so FK constraints can be satisfied when inserting real records.

In [7]:
cur.executescript('''
INSERT OR IGNORE INTO Type (type_name) VALUES
    ('Book'), ('DVD'), ('Magazine'), ('Audiobook'), ('E-Book');

INSERT OR IGNORE INTO Acquisition_Status (status_name) VALUES
    ('Not Yet Ordered'), ('Ordered'), ('Acquired');

INSERT OR IGNORE INTO Copy_Status (status_name) VALUES
    ('Available'), ('Checked Out'), ('Lost'), ('Damaged'), ('Under Repair');

INSERT OR IGNORE INTO Member_Status (status_name) VALUES
    ('Active'), ('Suspended'), ('Expired');

INSERT OR IGNORE INTO Request_Status (status_name) VALUES
    ('Open'), ('In Progress'), ('Resolved'), ('Closed');

INSERT OR IGNORE INTO Audience (audience_type) VALUES
    ('Children'), ('Teens'), ('Adults'), ('Seniors'), ('General');
''')
con.commit()
print('Reference data seeded.')

Reference data seeded.


## 7 — Verification
Confirm all tables exist and FKs are enforced.

In [8]:
tables = cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall()
print(f'{len(tables)} tables found:')
for (t,) in tables:
    print(' ', t)

24 tables found:
  Acquisition_Status
  Audience
  Author
  Copy_Status
  Donates
  Event
  Fine
  Help_Request
  Item
  Item_Copy
  Library
  Loan
  Member
  Member_Status
  Payment
  Personnel
  Registers_For
  Request_Status
  Room
  Targets
  Type
  Volunteer
  Written_By
  sqlite_sequence


In [9]:
# Confirm FK enforcement is on for this connection
fk_status = cur.execute('PRAGMA foreign_keys;').fetchone()[0]
print('foreign_keys PRAGMA =', fk_status, '(1 = enabled)')

# List triggers
triggers = cur.execute(
    "SELECT name, tbl_name FROM sqlite_master WHERE type='trigger' ORDER BY tbl_name;"
).fetchall()
print(f'\n{len(triggers)} triggers:')
for name, tbl in triggers:
    print(f'  {name}  →  {tbl}')

foreign_keys PRAGMA = 1 (1 = enabled)

8 triggers:
  trg_event_no_room_double_book  →  Event
  trg_loan_insert_copy_status  →  Loan
  trg_loan_return_copy_status  →  Loan
  trg_loan_prevent_unavailable  →  Loan
  trg_loan_prevent_suspended_member  →  Loan
  trg_payment_no_overpay  →  Payment
  trg_event_capacity_check  →  Registers_For
  trg_volunteer_must_be_member  →  Volunteer


In [10]:
# Quick FK integrity check on the seeded reference data
fk_violations = cur.execute('PRAGMA foreign_key_check;').fetchall()
if fk_violations:
    print('FK violations found:', fk_violations)
else:
    print('No FK violations — schema integrity confirmed.')

con.close()
print('Connection closed.')

No FK violations — schema integrity confirmed.
Connection closed.
